# Agentic HITL on kind — two approval gates for MCP agents

One MCP server, two human-in-the-loop gates:

- **Agent HITL**: the end user approves dangerous actions in the kagent chat.
- **Gateway HITL**: a platform reviewer approves privileged cross-team actions in a separate approval queue.

Same agent, same MCP wire protocol. The difference is **who approves** and **where the approval happens**.

```mermaid
flowchart LR
  User[End user<br/>kagent chat] --> Agent[kagent agent<br/>dba-assistant]
  Reviewer[Platform reviewer<br/>approval queue] --> UI[hitl-ui<br/>localhost:38090]

  Agent --> Pub[RemoteMCPServer<br/>ops-tools-public]
  Agent --> Priv[RemoteMCPServer<br/>ops-tools-privileged]

  Pub --> GW[agentgateway<br/>hitl-gateway]
  Priv --> GW

  GW -->|/public/mcp<br/>no gateway gate| PublicMCP[ops-tools /public<br/>cluster_db_query<br/>truncate_table]
  GW -->|/privileged/mcp<br/>extAuth parks request| ExtAuth[hitl-extauth<br/>gRPC Check waits]
  ExtAuth --> PrivMCP[ops-tools /privileged<br/>run_migration]
  UI -.approve / reject.-> ExtAuth

  Agent -.agent HITL<br/>requireApproval.-> User

  classDef user fill:#DBEAFE,stroke:#3B82F6,color:#1E3A8A,stroke-width:2px
  classDef gw fill:#7C3AED,color:#fff,stroke:#5B21B6,stroke-width:3px
  classDef mcp fill:#D1FAE5,stroke:#10B981,color:#064E3B,stroke-width:2px
  classDef hitl fill:#FEF3C7,stroke:#F59E0B,color:#78350F,stroke-width:2px
  class User,Reviewer user
  class Agent,GW gw
  class Pub,Priv,PublicMCP,PrivMCP mcp
  class UI,ExtAuth hitl
```

**Demo scenes:**

1. `cluster_db_query` — safe read, no approvals.
2. `truncate_table` — destructive user-owned action, approval card in kagent chat.
3. `run_migration` — privileged platform action, approval card in the platform queue.
4. Reject a gateway approval and show the denial returning to the agent.

## Setup

This notebook expects `./scripts/quick.sh up` to have completed in this folder. It does **not** install or tear down the lab; it only surfaces the manifests, opens port-forwards, and gives you helper commands while you drive the browser demo.

Main URLs after the port-forward cell:

- kagent dashboard: `http://localhost:38080`
- HITL approval queue: `http://localhost:38090`
- ext-auth admin API: `http://localhost:38091`
- ops-tools state API: `http://localhost:38082/state`

In [ ]:
export KUBECTL_CONTEXT=kind-hitl

kubectl --context $KUBECTL_CONTEXT cluster-info | head -1

echo
echo '--- key workloads ---'
for ns in agentgateway-system kagent ops-tools hitl; do
  echo "
# $ns"
  kubectl --context $KUBECTL_CONTEXT -n "$ns" get deploy,pod 2>/dev/null || true
done

echo
echo '--- gateway / routes / policy / agents ---'
kubectl --context $KUBECTL_CONTEXT get gateway,httproute,agentgatewaypolicy,remotemcpserver,agent -A 2>/dev/null

## Port-forwards

Start the two browser surfaces plus two helper APIs. Keep this cell's background port-forwards running while you walk the demo.

In [ ]:
# Clear stale forwards from previous demo runs.
pkill -f 'port-forward.*38080:8080' 2>/dev/null || true
pkill -f 'port-forward.*38090:80' 2>/dev/null || true
pkill -f 'port-forward.*38091:8081' 2>/dev/null || true
pkill -f 'port-forward.*38082:8080' 2>/dev/null || true
sleep 1

kubectl --context $KUBECTL_CONTEXT -n kagent    port-forward svc/kagent-ui 38080:8080 >/tmp/hitl-kagent-ui-pf.log 2>&1 &
kubectl --context $KUBECTL_CONTEXT -n hitl      port-forward svc/hitl-ui 38090:80 >/tmp/hitl-ui-pf.log 2>&1 &
kubectl --context $KUBECTL_CONTEXT -n hitl      port-forward svc/hitl-extauth-admin 38091:8081 >/tmp/hitl-admin-pf.log 2>&1 &
kubectl --context $KUBECTL_CONTEXT -n ops-tools port-forward svc/ops-tools 38082:8080 >/tmp/hitl-ops-pf.log 2>&1 &

for f in /tmp/hitl-kagent-ui-pf.log /tmp/hitl-ui-pf.log /tmp/hitl-admin-pf.log /tmp/hitl-ops-pf.log; do
  for _ in $(seq 1 10); do grep -q 'Forwarding from' "$f" 2>/dev/null && break; sleep 1; done
  echo "$(basename "$f"): $(head -1 "$f")"
done

echo
echo 'Open:'
echo '  kagent dashboard      http://localhost:38080'
echo '  HITL approval queue   http://localhost:38090'
echo '  ext-auth pending API  http://localhost:38091/pending'
echo '  ops-tools state       http://localhost:38082/state' 

## 1. What got deployed

The topology is intentionally simple:

- One `ops-tools` MCP server process exposes two Streamable HTTP MCP endpoints: `/public/mcp` and `/privileged/mcp`.
- Two kagent `RemoteMCPServer` objects point at those two paths through agentgateway.
- The `/public` route has no gateway HITL.
- The `/privileged` route has an `AgentgatewayPolicy` with `extAuth`, so every real `tools/call` parks at `hitl-extauth` until a platform reviewer decides.

In [ ]:
show() {
  local ns="$1" kind="$2" name="$3"
  echo "
===== $ns/$kind/$name ====="
  kubectl --context $KUBECTL_CONTEXT -n "$ns" get "$kind" "$name" -o yaml     | yq 'del(
        .metadata.managedFields,
        .metadata.creationTimestamp,
        .metadata.uid,
        .metadata.resourceVersion,
        .metadata.generation,
        .metadata.annotations."kubectl.kubernetes.io/last-applied-configuration",
        .status
      )'
}

show agentgateway-system gateway hitl-gateway
show ops-tools httproute mcp-public
show ops-tools httproute mcp-privileged
show ops-tools agentgatewaypolicy privileged-extauth
show kagent remotemcpserver ops-tools-public
show kagent remotemcpserver ops-tools-privileged
show kagent agent dba-assistant
show kagent agent dba-assistant-langgraph

## 2. The MCP server split

The gateway does not inspect tool names to decide which approval tier applies. The MCP server exposes two URL paths:

- `/public/mcp`: `cluster_db_query`, `truncate_table`
- `/privileged/mcp`: `run_migration`

That makes the platform gate visible in the Gateway API topology instead of hiding it in body-inspection logic.

In [ ]:
sed -n '55,127p' src/ops-tools/server.py

## 3. Scene 1 — safe read, no approvals

In the kagent dashboard (`http://localhost:38080`), select `dba-assistant` or `dba-assistant-langgraph`, then ask:

```text
What's in the orders table?
```

Expected behavior:

- The agent calls `cluster_db_query` through `/public/mcp`.
- No approval card appears in the chat.
- The platform queue at `http://localhost:38090` stays empty.
- The agent prints the mock orders rows and current schema version (`v2`).

In [ ]:
echo 'Current mock DB state:'
curl -sS http://localhost:38082/state | jq

echo
echo 'Gateway approval queue should be empty:'
curl -sS http://localhost:38091/pending | jq

## 4. Scene 2 — agent HITL in the chat

In the same kagent chat, ask:

```text
Truncate the orders table.
```

Expected behavior:

- The agent chooses `truncate_table`.
- kagent renders an approval card **inside the chat**.
- The platform queue stays empty because this is an end-user decision, not a platform decision.
- Click **Approve** in the chat. The table becomes empty.

This is configured by `requireApproval: [truncate_table]` on the declarative agent, or by LangGraph `interrupt()` in the BYO variant.

In [ ]:
echo 'Agent-side HITL config:'
kubectl --context $KUBECTL_CONTEXT -n kagent get agent dba-assistant -o yaml   | yq '.spec.declarative.tools[] | select(.mcpServer.name == "ops-tools-public") | .mcpServer'

echo
echo 'After approving truncate_table in the chat, row_counts.orders should be 0:'
curl -sS http://localhost:38082/state | jq '{schema_version, row_counts, audit}'

echo
echo 'Platform queue should still be empty:'
curl -sS http://localhost:38091/pending | jq

## 5. Scene 3 — gateway HITL in the platform queue

Now ask the agent:

```text
Apply migration v3.
```

Expected behavior:

- The agent calls `run_migration` through `/privileged/mcp`.
- kagent shows the tool call as executing / waiting.
- No chat approval card appears this time.
- The platform queue at `http://localhost:38090` gets a card.
- Approve it in the queue, or use the helper cell below.

The agent does not know about this gate. Agentgateway parked the HTTP request in `hitl-extauth` before it reached the MCP server.

In [ ]:
echo 'Pending gateway approvals:'
curl -sS http://localhost:38091/pending | jq

echo
echo 'ext-auth parked-call logs:'
kubectl --context $KUBECTL_CONTEXT -n hitl logs deploy/hitl-extauth --tail=30   | grep -E '\[parked\]|\[approved\]|\[denied\]|\[passthrough\]' || true

### Approve the first pending gateway request

Use the browser UI for the real demo. This helper is useful if you want to drive the approval from the notebook.

In [ ]:
PENDING_ID=$(curl -sS http://localhost:38091/pending | jq -r '.pending[0].id // empty')
if [ -z "$PENDING_ID" ]; then
  echo 'No pending gateway request.'
else
  echo "Approving $PENDING_ID"
  curl -sS -X POST "http://localhost:38091/decide/$PENDING_ID"     -H 'Content-Type: application/json'     -d '{"approved":true,"reason":"approved from notebook"}'     -o /dev/null -w 'HTTP %{http_code}
'
fi

echo
echo 'DB state after approval:'
curl -sS http://localhost:38082/state | jq '{schema_version, row_counts, audit}' 

## 6. Scene 4 — gateway rejection path

Ask the agent again:

```text
Apply migration v4.
```

When the card appears in the platform queue, reject it in the UI or use the helper cell below.

Expected behavior:

- The agent's tool call unblocks with an HTTP 403-style denial.
- The migration does not happen.
- The agent surfaces the rejection reason and does not retry.

In [ ]:
PENDING_ID=$(curl -sS http://localhost:38091/pending | jq -r '.pending[0].id // empty')
if [ -z "$PENDING_ID" ]; then
  echo 'No pending gateway request.'
else
  echo "Rejecting $PENDING_ID"
  curl -sS -X POST "http://localhost:38091/decide/$PENDING_ID"     -H 'Content-Type: application/json'     -d '{"approved":false,"reason":"rejected by platform reviewer from notebook"}'     -o /dev/null -w 'HTTP %{http_code}
'
fi

echo
echo 'DB state after rejection:'
curl -sS http://localhost:38082/state | jq '{schema_version, row_counts, audit}' 

## 7. Inspect while running

Useful when the browser demo is mid-flight.

In [ ]:
echo '--- ops-tools state ---'
curl -sS http://localhost:38082/state | jq

echo
echo '--- pending gateway approvals ---'
curl -sS http://localhost:38091/pending | jq

echo
echo '--- ext-auth recent log ---'
kubectl --context $KUBECTL_CONTEXT -n hitl logs deploy/hitl-extauth --tail=40   | grep -E '\[parked\]|\[approved\]|\[denied\]|\[passthrough\]' || true

## Wrap-up

The demo separates two different approval problems:

- **Agent HITL** protects the end user from their own agent doing something destructive. The approval belongs in the conversation.
- **Gateway HITL** protects shared infrastructure from any agent. The approval belongs at the platform boundary, with a queue and audit trail.

The important point is not that both pause a tool call. The important point is that they pause it for **different humans**.

## Cleanup (optional)

This stops notebook port-forwards only. It leaves the `kind-hitl` cluster running so you can rerun the demo. Full teardown is `./scripts/quick.sh teardown`.

In [ ]:
pkill -f 'port-forward.*38080:8080' 2>/dev/null || true
pkill -f 'port-forward.*38090:80' 2>/dev/null || true
pkill -f 'port-forward.*38091:8081' 2>/dev/null || true
pkill -f 'port-forward.*38082:8080' 2>/dev/null || true
echo 'port-forwards stopped' 